# I/O and Interoperability

This notebook introduces the main data-export and backend-conversion paths.

AnnNet is the source of truth. External tables and backend graphs are projections of that source of truth.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / 'annnet').exists():
    for parent in repo_root.parents:
        if (parent / 'annnet').exists():
            repo_root = parent
            break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import annnet as an


In [ ]:
G = an.AnnNet(directed=True)
G.add_vertices([
    ('EGFR', {'kind': 'protein'}),
    ('GRB2', {'kind': 'protein'}),
    ('SOS1', {'kind': 'protein'}),
    ('RAS', {'kind': 'protein'}),
])
G.add_edges('EGFR', 'GRB2', edge_id='e1', confidence=0.99)
G.add_edges('GRB2', 'SOS1', edge_id='e2', confidence=0.95)
G.add_edges(src=['SOS1', 'RAS', 'EGFR'], edge_id='h1', directed=False, process='complex')


## Export to explicit tables

`to_dataframes(...)` is the easiest way to make the graph explicit as separate tables.


In [ ]:
tables = an.to_dataframes(G)
print(sorted(tables))
print('nodes table:')
print(tables['nodes'])
print('edges table:')
print(tables['edges'])
print('hyperedges table:')
print(tables['hyperedges'])


## Backend conversion

The lazy backend proxies live on the graph object itself: `G.nx`, `G.ig`, and `G.gt`.

Use `backend()` when you want the concrete projected graph object.


In [ ]:
nx_graph = G.nx.backend()
print(type(nx_graph).__name__)
print('networkx nodes / edges:', nx_graph.number_of_nodes(), nx_graph.number_of_edges())


## Native round-trip

The native `.annnet` format is the high-fidelity persistence format. Use it when AnnNet is the system of record.


In [ ]:
from pathlib import Path
out = Path('tmp_tutorial_graph.annnet')
G.write(out, overwrite=True)
G2 = an.AnnNet.read(out)
print('round-trip shape:', G2.shape)
print('round-trip vertices:', G2.vertices())
